<a href="https://colab.research.google.com/github/sdg331/BDA/blob/main/11w_ch06_dataframe_group_merge.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 11주차 데이터프레임 그룹화와 결합 실습

## 0. 기본 준비

In [1]:
import pandas as pd
import numpy as np
from IPython.display import display_html

pd.set_option('display.precision', 2)
pd.set_option('display.max_rows', 30)
pd.set_option('display.max_columns', 20)

In [2]:
# 표 여러 개를 가로로 확인하기
def show_tables(*tables):
    html = ""
    for table in tables:
        html += table.to_html() + "&nbsp;" * 4
    display_html(html.replace("table", "table style='display:inline'"), raw=True)

## 1. 실습 데이터 만들기

In [3]:
# 카페 주문 데이터
orders = pd.DataFrame({
    "order_id": [101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112],
    "store": ["강남", "강남", "홍대", "홍대", "신촌", "신촌", "강남", "홍대", "신촌", "강남", "홍대", "신촌"],
    "menu": ["아메리카노", "라떼", "아메리카노", "케이크", "라떼", "티", "케이크", "티", "아메리카노", "티", "라떼", "케이크"],
    "category": ["coffee", "coffee", "coffee", "dessert", "coffee", "tea", "dessert", "tea", "coffee", "tea", "coffee", "dessert"],
    "qty": [2, 1, 3, 2, 2, 1, 1, 4, 2, 3, 1, 2],
    "price": [4500, 5500, 4500, 6500, 5500, 5000, 6500, 5000, 4500, 5000, 5500, 6500],
    "member": ["Y", "N", "Y", "Y", "N", "Y", "N", "Y", "Y", "N", "Y", "N"]
})
orders

,order_id,store,menu,category,qty,price,member
0,101,강남,아메리카노,coffee,2,4500,Y
1,102,강남,라떼,coffee,1,5500,N
2,103,홍대,아메리카노,coffee,3,4500,Y
3,104,홍대,케이크,dessert,2,6500,Y
4,105,신촌,라떼,coffee,2,5500,N
5,106,신촌,티,tea,1,5000,Y
6,107,강남,케이크,dessert,1,6500,N
7,108,홍대,티,tea,4,5000,Y
8,109,신촌,아메리카노,coffee,2,4500,Y
9,110,강남,티,tea,3,5000,N


In [4]:
# 매출액 열 추가
orders["amount"] = orders["qty"] * orders["price"]
orders

,order_id,store,menu,category,qty,price,member,amount
0,101,강남,아메리카노,coffee,2,4500,Y,9000
1,102,강남,라떼,coffee,1,5500,N,5500
2,103,홍대,아메리카노,coffee,3,4500,Y,13500
3,104,홍대,케이크,dessert,2,6500,Y,13000
4,105,신촌,라떼,coffee,2,5500,N,11000
5,106,신촌,티,tea,1,5000,Y,5000
6,107,강남,케이크,dessert,1,6500,N,6500
7,108,홍대,티,tea,4,5000,Y,20000
8,109,신촌,아메리카노,coffee,2,4500,Y,9000
9,110,강남,티,tea,3,5000,N,15000


In [5]:
# 데이터 기본 정보
orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12 entries, 0 to 11
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   order_id  12 non-null     int64 
 1   store     12 non-null     object
 2   menu      12 non-null     object
 3   category  12 non-null     object
 4   qty       12 non-null     int64 
 5   price     12 non-null     int64 
 6   member    12 non-null     object
 7   amount    12 non-null     int64 
dtypes: int64(4), object(4)
memory usage: 900.0+ bytes


In [6]:
# 수치형 요약
orders.describe()

,order_id,qty,price,amount
count,12.00,12.00,12.00,12.00
mean,106.50,2.00,5375.00,10500.00
std,3.61,0.95,772.39,4602.37
min,101.00,1.00,4500.00,5000.00
25%,103.75,1.00,4875.00,6250.00
50%,106.50,2.00,5250.00,10000.00
75%,109.25,2.25,5750.00,13125.00
max,112.00,4.00,6500.00,20000.00


In [7]:
# 문자형 포함 요약
orders.describe(include="all")

,order_id,store,menu,category,qty,price,member,amount
count,12.00,12,12,12,12.00,12.00,12,12.00
unique,NaN,3,4,3,NaN,NaN,2,NaN
top,NaN,강남,아메리카노,coffee,NaN,NaN,Y,NaN
freq,NaN,4,3,6,NaN,NaN,7,NaN
mean,106.50,NaN,NaN,NaN,2.00,5375.00,NaN,10500.00
std,3.61,NaN,NaN,NaN,0.95,772.39,NaN,4602.37
min,101.00,NaN,NaN,NaN,1.00,4500.00,NaN,5000.00
25%,103.75,NaN,NaN,NaN,1.00,4875.00,NaN,6250.00
50%,106.50,NaN,NaN,NaN,2.00,5250.00,NaN,10000.00
75%,109.25,NaN,NaN,NaN,2.25,5750.00,NaN,13125.00


## 2. agg() 기본 사용

In [8]:
# 수치형 열 평균
orders[["qty", "price", "amount"]].agg("mean")

,0
qty,2.0
price,5375.0
amount,10500.0


In [9]:
# 수치형 열 합계
orders[["qty", "price", "amount"]].agg("sum")

,0
qty,24
price,64500
amount,126000


In [10]:
# 여러 집계 함수 한 번에 사용
orders[["qty", "price", "amount"]].agg(["count", "sum", "mean", "min", "max"])

,qty,price,amount
count,12.0,12.0,12.0
sum,24.0,64500.0,126000.0
mean,2.0,5375.0,10500.0
min,1.0,4500.0,5000.0
max,4.0,6500.0,20000.0


In [11]:
# 행 방향 집계
orders[["qty", "price"]].agg("sum", axis=1)

,0
0,4502
1,5501
2,4503
3,6502
4,5502
5,5001
6,6501
7,5004
8,4502
9,5003


In [12]:
# 행 방향 평균
orders[["qty", "price", "amount"]].agg("mean", axis=1)

,0
0,4500.67
1,3667.00
2,6001.00
3,6500.67
4,5500.67
5,3333.67
6,4333.67
7,8334.67
8,4500.67
9,6667.67


In [13]:
# 특정 열에 서로 다른 집계 함수 적용
orders.agg({
    "qty": ["sum", "mean", "max"],
    "amount": ["sum", "mean", "max"],
    "menu": ["nunique"]
})

,qty,amount,menu
sum,24.0,126000.0,NaN
mean,2.0,10500.0,NaN
max,4.0,20000.0,NaN
nunique,NaN,NaN,4.0


In [14]:
# 집계 결과를 데이터프레임으로 정리
summary_basic = pd.DataFrame({
    "지표": ["주문건수", "총수량", "총매출", "평균매출"],
    "값": [orders["order_id"].count(), orders["qty"].sum(), orders["amount"].sum(), orders["amount"].mean()]
})
summary_basic

,지표,값
0,주문건수,12.0
1,총수량,24.0
2,총매출,126000.0
3,평균매출,10500.0


## 3. groupby()로 집단별 요약

In [15]:
# 지점별 평균
orders.groupby("store")[["qty", "amount"]].mean()

,qty,amount
store,,
강남,1.75,9000.0
신촌,1.75,9500.0
홍대,2.50,13000.0


In [16]:
# 지점별 합계
orders.groupby("store")[["qty", "amount"]].sum()

,qty,amount
store,,
강남,7,36000
신촌,7,38000
홍대,10,52000


In [17]:
# 메뉴별 주문 수량 합계
orders.groupby("menu")[["qty"]].sum()

,qty
menu,
라떼,4
아메리카노,7
케이크,5
티,8


In [18]:
# 메뉴별 매출 합계
orders.groupby("menu")[["amount"]].sum()

,amount
menu,
라떼,22000
아메리카노,31500
케이크,32500
티,40000


In [19]:
# 메뉴별 여러 통계량
orders.groupby("menu")["amount"].agg(["count", "sum", "mean", "min", "max"])

,count,sum,mean,min,max
menu,,,,,
라떼,3,22000,7333.33,5500,11000
아메리카노,3,31500,10500.00,9000,13500
케이크,3,32500,10833.33,6500,13000
티,3,40000,13333.33,5000,20000


In [20]:
# 새 열 이름 지정
orders.groupby("store").agg(
    주문건수=("order_id", "count"),
    총수량=("qty", "sum"),
    총매출=("amount", "sum"),
    평균매출=("amount", "mean")
)

,주문건수,총수량,총매출,평균매출
store,,,,
강남,4,7,36000,9000.0
신촌,4,7,38000,9500.0
홍대,4,10,52000,13000.0


In [21]:
# as_index=False 사용
orders.groupby("store", as_index=False).agg(
    주문건수=("order_id", "count"),
    총수량=("qty", "sum"),
    총매출=("amount", "sum")
)

,store,주문건수,총수량,총매출
0,강남,4,7,36000
1,신촌,4,7,38000
2,홍대,4,10,52000


In [22]:
# 카테고리별 요약
orders.groupby("category").agg(
    주문건수=("order_id", "count"),
    총매출=("amount", "sum"),
    평균단가=("price", "mean")
)

,주문건수,총매출,평균단가
category,,,
coffee,6,53500,5000.0
dessert,3,32500,6500.0
tea,3,40000,5000.0


In [23]:
# 회원 여부별 요약
orders.groupby("member").agg(
    주문건수=("order_id", "count"),
    총수량=("qty", "sum"),
    총매출=("amount", "sum")
)

,주문건수,총수량,총매출
member,,,
N,5,9,51000
Y,7,15,75000


## 4. 여러 기준으로 groupby()

In [24]:
# 지점과 카테고리별 매출
orders.groupby(["store", "category"]).agg(
    주문건수=("order_id", "count"),
    총매출=("amount", "sum")
)

주문건수    총매출
store category             
강남    coffee       2  14500
      dessert      1   6500
      tea          1  15000
신촌    coffee       2  20000
      dessert      1  13000
      tea          1   5000
홍대    coffee       2  19000
      dessert      1  13000
      tea          1  20000

In [25]:
# 지점과 메뉴별 수량
orders.groupby(["store", "menu"]).agg(
    총수량=("qty", "sum"),
    총매출=("amount", "sum")
)

총수량    총매출
store menu             
강남    라떼       1   5500
      아메리카노    2   9000
      케이크      1   6500
      티        3  15000
신촌    라떼       2  11000
      아메리카노    2   9000
      케이크      2  13000
      티        1   5000
홍대    라떼       1   5500
      아메리카노    3  13500
      케이크      2  13000
      티        4  20000

In [26]:
# 인덱스를 열로 되돌리기
store_menu = orders.groupby(["store", "menu"]).agg(
    총수량=("qty", "sum"),
    총매출=("amount", "sum")
).reset_index()

store_menu

,store,menu,총수량,총매출
0,강남,라떼,1,5500
1,강남,아메리카노,2,9000
2,강남,케이크,1,6500
3,강남,티,3,15000
4,신촌,라떼,2,11000
5,신촌,아메리카노,2,9000
6,신촌,케이크,2,13000
7,신촌,티,1,5000
8,홍대,라떼,1,5500
9,홍대,아메리카노,3,13500


In [27]:
# 지점별 매출 높은 순서
store_menu.sort_values(["store", "총매출"], ascending=[True, False])

,store,menu,총수량,총매출
3,강남,티,3,15000
1,강남,아메리카노,2,9000
2,강남,케이크,1,6500
0,강남,라떼,1,5500
6,신촌,케이크,2,13000
4,신촌,라떼,2,11000
5,신촌,아메리카노,2,9000
7,신촌,티,1,5000
11,홍대,티,4,20000
9,홍대,아메리카노,3,13500


In [28]:
# 전체에서 매출 높은 순서
store_menu.sort_values("총매출", ascending=False)

,store,menu,총수량,총매출
11,홍대,티,4,20000
3,강남,티,3,15000
9,홍대,아메리카노,3,13500
6,신촌,케이크,2,13000
10,홍대,케이크,2,13000
4,신촌,라떼,2,11000
1,강남,아메리카노,2,9000
5,신촌,아메리카노,2,9000
2,강남,케이크,1,6500
0,강남,라떼,1,5500


## 5. groupby 객체 확인

In [29]:
# groupby 객체 생성
grouped_store = orders.groupby("store")
grouped_store

In [30]:
# groupby 객체 타입
type(grouped_store)

pandas.core.groupby.generic.DataFrameGroupBy

In [31]:
# 그룹 이름 확인
grouped_store.groups

{'강남': [0, 1, 6, 9], '신촌': [4, 5, 8, 11], '홍대': [2, 3, 7, 10]}

In [32]:
# 그룹을 반복문으로 확인
for key, value in grouped_store:
    print("그룹:", key)
    print(value)
    print()

그룹: 강남
   order_id store   menu category  qty  price member  amount
0       101    강남  아메리카노   coffee    2   4500      Y    9000
1       102    강남     라떼   coffee    1   5500      N    5500
6       107    강남    케이크  dessert    1   6500      N    6500
9       110    강남      티      tea    3   5000      N   15000

그룹: 신촌
    order_id store   menu category  qty  price member  amount
4        105    신촌     라떼   coffee    2   5500      N   11000
5        106    신촌      티      tea    1   5000      Y    5000
8        109    신촌  아메리카노   coffee    2   4500      Y    9000
11       112    신촌    케이크  dessert    2   6500      N   13000

그룹: 홍대
    order_id store   menu category  qty  price member  amount
2        103    홍대  아메리카노   coffee    3   4500      Y   13500
3        104    홍대    케이크  dessert    2   6500      Y   13000
7        108    홍대      티      tea    4   5000      Y   20000
10       111    홍대     라떼   coffee    1   5500      Y    5500



In [33]:
# 특정 그룹 가져오기
grouped_store.get_group("강남")

,order_id,store,menu,category,qty,price,member,amount
0,101,강남,아메리카노,coffee,2,4500,Y,9000
1,102,강남,라떼,coffee,1,5500,N,5500
6,107,강남,케이크,dessert,1,6500,N,6500
9,110,강남,티,tea,3,5000,N,15000


In [34]:
# 그룹별 첫 행
grouped_store.first()

,order_id,menu,category,qty,price,member,amount
store,,,,,,,
강남,101,아메리카노,coffee,2,4500,Y,9000
신촌,105,라떼,coffee,2,5500,N,11000
홍대,103,아메리카노,coffee,3,4500,Y,13500


In [35]:
# 그룹별 마지막 행
grouped_store.last()

,order_id,menu,category,qty,price,member,amount
store,,,,,,,
강남,110,티,tea,3,5000,N,15000
신촌,112,케이크,dessert,2,6500,N,13000
홍대,111,라떼,coffee,1,5500,Y,5500


In [36]:
# 그룹별 크기
grouped_store.size()

,0
store,
강남,4
신촌,4
홍대,4


## 6. value_counts()와 빈도 계산

In [37]:
# 메뉴 빈도
orders["menu"].value_counts()

,count
menu,
아메리카노,3
라떼,3
케이크,3
티,3


In [38]:
# 지점 빈도
orders["store"].value_counts()

,count
store,
강남,4
홍대,4
신촌,4


In [39]:
# 카테고리와 회원 여부 조합 빈도
orders[["category", "member"]].value_counts()

category  member
coffee    Y         4
          N         2
dessert   N         2
tea       Y         2
dessert   Y         1
tea       N         1
Name: count, dtype: int64

In [40]:
# 데이터프레임으로 변환
orders["menu"].value_counts().to_frame("주문건수")

,주문건수
menu,
아메리카노,3
라떼,3
케이크,3
티,3


In [41]:
# 빈도 조건 검색
orders["menu"].value_counts().to_frame("주문건수").query("주문건수 >= 3")

,주문건수
menu,
아메리카노,3
라떼,3
케이크,3
티,3


In [42]:
# groupby로 빈도 계산
orders.groupby("menu").agg(주문건수=("menu", "count"))

,주문건수
menu,
라떼,3
아메리카노,3
케이크,3
티,3


In [43]:
# groupby 결과에서 조건 검색
orders.groupby("menu").agg(주문건수=("menu", "count")).query("주문건수 >= 3")

,주문건수
menu,
라떼,3
아메리카노,3
케이크,3
티,3


## 7. assign()과 함수 체이닝

In [44]:
# 주문 데이터에 할인 적용
orders.assign(
    discount=np.where(orders["member"] == "Y", orders["amount"] * 0.1, 0)
)

,order_id,store,menu,category,qty,price,member,amount,discount
0,101,강남,아메리카노,coffee,2,4500,Y,9000,900.0
1,102,강남,라떼,coffee,1,5500,N,5500,0.0
2,103,홍대,아메리카노,coffee,3,4500,Y,13500,1350.0
3,104,홍대,케이크,dessert,2,6500,Y,13000,1300.0
4,105,신촌,라떼,coffee,2,5500,N,11000,0.0
5,106,신촌,티,tea,1,5000,Y,5000,500.0
6,107,강남,케이크,dessert,1,6500,N,6500,0.0
7,108,홍대,티,tea,4,5000,Y,20000,2000.0
8,109,신촌,아메리카노,coffee,2,4500,Y,9000,900.0
9,110,강남,티,tea,3,5000,N,15000,0.0


In [45]:
# 할인 후 결제금액 계산
orders_discount = orders.assign(
    discount=np.where(orders["member"] == "Y", orders["amount"] * 0.1, 0),
    pay_amount=lambda x: x["amount"] - x["discount"]
)
orders_discount

,order_id,store,menu,category,qty,price,member,amount,discount,pay_amount
0,101,강남,아메리카노,coffee,2,4500,Y,9000,900.0,8100.0
1,102,강남,라떼,coffee,1,5500,N,5500,0.0,5500.0
2,103,홍대,아메리카노,coffee,3,4500,Y,13500,1350.0,12150.0
3,104,홍대,케이크,dessert,2,6500,Y,13000,1300.0,11700.0
4,105,신촌,라떼,coffee,2,5500,N,11000,0.0,11000.0
5,106,신촌,티,tea,1,5000,Y,5000,500.0,4500.0
6,107,강남,케이크,dessert,1,6500,N,6500,0.0,6500.0
7,108,홍대,티,tea,4,5000,Y,20000,2000.0,18000.0
8,109,신촌,아메리카노,coffee,2,4500,Y,9000,900.0,8100.0
9,110,강남,티,tea,3,5000,N,15000,0.0,15000.0


In [46]:
# 지점별 할인 후 매출
orders_discount.groupby("store").agg(
    총매출=("amount", "sum"),
    할인금액=("discount", "sum"),
    결제금액=("pay_amount", "sum")
)

,총매출,할인금액,결제금액
store,,,
강남,36000,900.0,35100.0
신촌,38000,1400.0,36600.0
홍대,52000,5200.0,46800.0


In [47]:
# 체이닝으로 카테고리별 결제금액 요약
orders.assign(
    discount=np.where(orders["member"] == "Y", orders["amount"] * 0.1, 0),
    pay_amount=lambda x: x["amount"] - x["discount"]
).groupby("category").agg(
    주문건수=("order_id", "count"),
    결제금액=("pay_amount", "sum")
).sort_values("결제금액", ascending=False)

,주문건수,결제금액
category,,
coffee,6,49800.0
tea,3,37500.0
dessert,3,31200.0


## 8. pivot_table()로 요약표 만들기

In [48]:
# 지점별 카테고리 매출 피벗테이블
orders.pivot_table(
    index="store",
    columns="category",
    values="amount",
    aggfunc="sum",
    fill_value=0
)

category,coffee,dessert,tea
store,,,
강남,14500,6500,15000
신촌,20000,13000,5000
홍대,19000,13000,20000


In [49]:
# 지점별 메뉴 수량 피벗테이블
orders.pivot_table(
    index="store",
    columns="menu",
    values="qty",
    aggfunc="sum",
    fill_value=0
)

menu,라떼,아메리카노,케이크,티
store,,,,
강남,1,2,1,3
신촌,2,2,2,1
홍대,1,3,2,4


In [50]:
# margins=True로 합계 추가
orders.pivot_table(
    index="store",
    columns="category",
    values="amount",
    aggfunc="sum",
    fill_value=0,
    margins=True
)

category,coffee,dessert,tea,All
store,,,,
강남,14500,6500,15000,36000
신촌,20000,13000,5000,38000
홍대,19000,13000,20000,52000
All,53500,32500,40000,126000


## 9. concat()으로 세로 결합

In [51]:
# 1월 주문 데이터
jan = pd.DataFrame({
    "order_id": [201, 202, 203],
    "store": ["강남", "홍대", "신촌"],
    "amount": [12000, 15000, 9000]
})

# 2월 주문 데이터
feb = pd.DataFrame({
    "order_id": [204, 205, 206],
    "store": ["강남", "홍대", "신촌"],
    "amount": [18000, 13000, 17000]
})

show_tables(jan, feb)

,order_id,store,amount
0,201,강남,12000
1,202,홍대,15000
2,203,신촌,9000
,order_id,store,amount
0,204,강남,18000
1,205,홍대,13000
2,206,신촌,17000


In [52]:
# 세로 결합
pd.concat([jan, feb])

,order_id,store,amount
0,201,강남,12000
1,202,홍대,15000
2,203,신촌,9000
0,204,강남,18000
1,205,홍대,13000
2,206,신촌,17000


In [53]:
# 인덱스 재정리
pd.concat([jan, feb], ignore_index=True)

,order_id,store,amount
0,201,강남,12000
1,202,홍대,15000
2,203,신촌,9000
3,204,강남,18000
4,205,홍대,13000
5,206,신촌,17000


In [54]:
# 원래 인덱스를 열로 보존
pd.concat([jan, feb]).reset_index()

,index,order_id,store,amount
0,0,201,강남,12000
1,1,202,홍대,15000
2,2,203,신촌,9000
3,0,204,강남,18000
4,1,205,홍대,13000
5,2,206,신촌,17000


In [55]:
# 원래 인덱스를 버리고 초기화
pd.concat([jan, feb]).reset_index(drop=True)

,order_id,store,amount
0,201,강남,12000
1,202,홍대,15000
2,203,신촌,9000
3,204,강남,18000
4,205,홍대,13000
5,206,신촌,17000


In [56]:
# keys를 사용해 월 구분
pd.concat([jan, feb], keys=["1월", "2월"])

order_id store  amount
1월 0       201    강남   12000
   1       202    홍대   15000
   2       203    신촌    9000
2월 0       204    강남   18000
   1       205    홍대   13000
   2       206    신촌   17000

## 10. concat()으로 가로 결합

In [57]:
# 같은 행 순서라고 가정한 데이터
customer = pd.DataFrame({
    "customer_id": [1, 2, 3],
    "name": ["김지민", "이서연", "박민수"]
})

grade = pd.DataFrame({
    "grade": ["Gold", "Silver", "Bronze"],
    "point": [3200, 1800, 900]
})

show_tables(customer, grade)

,customer_id,name
0,1,김지민
1,2,이서연
2,3,박민수
,grade,point
0,Gold,3200
1,Silver,1800
2,Bronze,900


In [58]:
# 열 방향 결합
pd.concat([customer, grade], axis=1)

,customer_id,name,grade,point
0,1,김지민,Gold,3200
1,2,이서연,Silver,1800
2,3,박민수,Bronze,900


In [59]:
# 인덱스가 다를 때 가로 결합
grade2 = grade.copy()
grade2.index = [1, 2, 3]

show_tables(customer, grade2)

,customer_id,name
0,1,김지민
1,2,이서연
2,3,박민수
,grade,point
1,Gold,3200
2,Silver,1800
3,Bronze,900


In [60]:
# 인덱스 기준으로 붙기 때문에 NaN 발생
pd.concat([customer, grade2], axis=1)

,customer_id,name,grade,point
0,1.0,김지민,NaN,NaN
1,2.0,이서연,Gold,3200.0
2,3.0,박민수,Silver,1800.0
3,NaN,NaN,Bronze,900.0


In [61]:
# 인덱스를 다시 맞춘 뒤 결합
pd.concat([customer.reset_index(drop=True), grade2.reset_index(drop=True)], axis=1)

,customer_id,name,grade,point
0,1,김지민,Gold,3200
1,2,이서연,Silver,1800
2,3,박민수,Bronze,900


## 11. merge() 기본 사용

In [62]:
# 고객 정보
customers = pd.DataFrame({
    "customer_id": [1, 2, 3, 4],
    "name": ["김지민", "이서연", "박민수", "최하준"],
    "region": ["서울", "경기", "서울", "부산"]
})

# 구매 정보
purchases = pd.DataFrame({
    "customer_id": [1, 2, 2, 5],
    "product": ["노트북", "마우스", "키보드", "모니터"],
    "price": [1200000, 25000, 80000, 300000]
})

show_tables(customers, purchases)

,customer_id,name,region
0,1,김지민,서울
1,2,이서연,경기
2,3,박민수,서울
3,4,최하준,부산
,customer_id,product,price
0,1,노트북,1200000
1,2,마우스,25000
2,2,키보드,80000
3,5,모니터,300000


In [63]:
# 기본 merge
pd.merge(customers, purchases)

,customer_id,name,region,product,price
0,1,김지민,서울,노트북,1200000
1,2,이서연,경기,마우스,25000
2,2,이서연,경기,키보드,80000


In [64]:
# on 지정
pd.merge(customers, purchases, on="customer_id")

,customer_id,name,region,product,price
0,1,김지민,서울,노트북,1200000
1,2,이서연,경기,마우스,25000
2,2,이서연,경기,키보드,80000


In [65]:
# inner join
pd.merge(customers, purchases, on="customer_id", how="inner")

,customer_id,name,region,product,price
0,1,김지민,서울,노트북,1200000
1,2,이서연,경기,마우스,25000
2,2,이서연,경기,키보드,80000


In [66]:
# left join
pd.merge(customers, purchases, on="customer_id", how="left")

,customer_id,name,region,product,price
0,1,김지민,서울,노트북,1.20e+06
1,2,이서연,경기,마우스,2.50e+04
2,2,이서연,경기,키보드,8.00e+04
3,3,박민수,서울,NaN,NaN
4,4,최하준,부산,NaN,NaN


In [67]:
# right join
pd.merge(customers, purchases, on="customer_id", how="right")

,customer_id,name,region,product,price
0,1,김지민,서울,노트북,1200000
1,2,이서연,경기,마우스,25000
2,2,이서연,경기,키보드,80000
3,5,NaN,NaN,모니터,300000


In [68]:
# outer join
pd.merge(customers, purchases, on="customer_id", how="outer")

,customer_id,name,region,product,price
0,1,김지민,서울,노트북,1.20e+06
1,2,이서연,경기,마우스,2.50e+04
2,2,이서연,경기,키보드,8.00e+04
3,3,박민수,서울,NaN,NaN
4,4,최하준,부산,NaN,NaN
5,5,NaN,NaN,모니터,3.00e+05


In [69]:
# indicator로 결합 상태 확인
pd.merge(customers, purchases, on="customer_id", how="outer", indicator=True)

,customer_id,name,region,product,price,_merge
0,1,김지민,서울,노트북,1.20e+06,both
1,2,이서연,경기,마우스,2.50e+04,both
2,2,이서연,경기,키보드,8.00e+04,both
3,3,박민수,서울,NaN,NaN,left_only
4,4,최하준,부산,NaN,NaN,left_only
5,5,NaN,NaN,모니터,3.00e+05,right_only


## 12. DataFrame 메서드 merge()

In [70]:
# pd.merge와 DataFrame.merge 비교
result1 = pd.merge(customers, purchases, on="customer_id", how="left")
result2 = customers.merge(purchases, on="customer_id", how="left")

show_tables(result1, result2)

,customer_id,name,region,product,price
0,1,김지민,서울,노트북,1.20e+06
1,2,이서연,경기,마우스,2.50e+04
2,2,이서연,경기,키보드,8.00e+04
3,3,박민수,서울,NaN,NaN
4,4,최하준,부산,NaN,NaN
,customer_id,name,region,product,price
0,1,김지민,서울,노트북,1.20e+06
1,2,이서연,경기,마우스,2.50e+04
2,2,이서연,경기,키보드,8.00e+04
3,3,박민수,서울,NaN,NaN


In [71]:
# 왼쪽 데이터 기준 결합
customers.merge(purchases, on="customer_id", how="left")

,customer_id,name,region,product,price
0,1,김지민,서울,노트북,1.20e+06
1,2,이서연,경기,마우스,2.50e+04
2,2,이서연,경기,키보드,8.00e+04
3,3,박민수,서울,NaN,NaN
4,4,최하준,부산,NaN,NaN


In [72]:
# 오른쪽 데이터 기준 결합
purchases.merge(customers, on="customer_id", how="left")

,customer_id,product,price,name,region
0,1,노트북,1200000,김지민,서울
1,2,마우스,25000,이서연,경기
2,2,키보드,80000,이서연,경기
3,5,모니터,300000,NaN,NaN


## 13. 서로 다른 키 이름으로 merge()

In [73]:
# 부서 정보
employees = pd.DataFrame({
    "emp_id": [10, 11, 12, 13],
    "emp_name": ["Ahn", "Bae", "Choi", "Do"],
    "dept_code": ["D1", "D2", "D1", "D3"]
})

departments = pd.DataFrame({
    "code": ["D1", "D2", "D4"],
    "dept_name": ["개발팀", "디자인팀", "마케팅팀"]
})

show_tables(employees, departments)

emp_id 
 emp_name 
 dept_code 
 
 
 
 
 0 
 10 
 Ahn 
 D1 
 
 
 1 
 11 
 Bae 
 D2 
 
 
 2 
 12 
 Choi 
 D1 
 
 
 3 
 13 
 Do 
 D3 
 
 
      
 
 
 
 code 
 dept_name 
 
 
 
 
 0 
 D1 
 개발팀 
 
 
 1 
 D2 
 디자인팀 
 
 
 2 
 D4 
 마케팅팀

In [74]:
# left_on, right_on 사용
employees.merge(departments, left_on="dept_code", right_on="code", how="left")

,emp_id,emp_name,dept_code,code,dept_name
0,10,Ahn,D1,D1,개발팀
1,11,Bae,D2,D2,디자인팀
2,12,Choi,D1,D1,개발팀
3,13,Do,D3,NaN,NaN


In [75]:
# 필요 없는 code 열 제거
employees.merge(departments, left_on="dept_code", right_on="code", how="left").drop(columns="code")

,emp_id,emp_name,dept_code,dept_name
0,10,Ahn,D1,개발팀
1,11,Bae,D2,디자인팀
2,12,Choi,D1,개발팀
3,13,Do,D3,NaN


In [76]:
# outer join으로 전체 확인
employees.merge(departments, left_on="dept_code", right_on="code", how="outer", indicator=True)

,emp_id,emp_name,dept_code,code,dept_name,_merge
0,10.0,Ahn,D1,D1,개발팀,both
1,12.0,Choi,D1,D1,개발팀,both
2,11.0,Bae,D2,D2,디자인팀,both
3,13.0,Do,D3,NaN,NaN,left_only
4,NaN,NaN,NaN,D4,마케팅팀,right_only


## 14. 중복 키가 있는 merge()

In [77]:
# 주문 헤더
order_header = pd.DataFrame({
    "order_id": [301, 302, 303],
    "customer": ["지민", "서연", "민수"]
})

# 주문 상세
order_detail = pd.DataFrame({
    "order_id": [301, 301, 302, 303, 303],
    "item": ["커피", "케이크", "라떼", "티", "스콘"],
    "qty": [2, 1, 1, 3, 2]
})

show_tables(order_header, order_detail)

order_id 
 customer 
 
 
 
 
 0 
 301 
 지민 
 
 
 1 
 302 
 서연 
 
 
 2 
 303 
 민수 
 
 
      
 
 
 
 order_id 
 item 
 qty 
 
 
 
 
 0 
 301 
 커피 
 2 
 
 
 1 
 301 
 케이크 
 1 
 
 
 2 
 302 
 라떼 
 1 
 
 
 3 
 303 
 티 
 3 
 
 
 4 
 303 
 스콘 
 2

In [78]:
# 1:N 구조 merge
order_header.merge(order_detail, on="order_id", how="left")

,order_id,customer,item,qty
0,301,지민,커피,2
1,301,지민,케이크,1
2,302,서연,라떼,1
3,303,민수,티,3
4,303,민수,스콘,2


In [79]:
# 주문별 품목 수
order_header.merge(order_detail, on="order_id", how="left").groupby("order_id").agg(
    고객명=("customer", "first"),
    품목수=("item", "count"),
    총수량=("qty", "sum")
)

,고객명,품목수,총수량
order_id,,,
301,지민,2,3
302,서연,1,1
303,민수,2,5


## 15. 공통 열 이름이 있을 때 suffixes

In [80]:
# 전년도 점수
score_2025 = pd.DataFrame({
    "id": [1, 2, 3],
    "name": ["지민", "서연", "민수"],
    "score": [80, 90, 75]
})

# 올해 점수
score_2026 = pd.DataFrame({
    "id": [1, 2, 4],
    "name": ["지민", "서연", "하준"],
    "score": [88, 95, 70]
})

show_tables(score_2025, score_2026)

,id,name,score
0,1,지민,80
1,2,서연,90
2,3,민수,75
,id,name,score
0,1,지민,88
1,2,서연,95
2,4,하준,70


In [81]:
# suffixes 미지정
score_2025.merge(score_2026, on="id", how="outer")

,id,name_x,score_x,name_y,score_y
0,1,지민,80.0,지민,88.0
1,2,서연,90.0,서연,95.0
2,3,민수,75.0,NaN,NaN
3,4,NaN,NaN,하준,70.0


In [82]:
# suffixes 지정
score_2025.merge(score_2026, on="id", how="outer", suffixes=("_2025", "_2026"))

,id,name_2025,score_2025,name_2026,score_2026
0,1,지민,80.0,지민,88.0
1,2,서연,90.0,서연,95.0
2,3,민수,75.0,NaN,NaN
3,4,NaN,NaN,하준,70.0


## 16. cross join

In [83]:
# 색상과 사이즈 조합 만들기
colors = pd.DataFrame({"color": ["black", "white", "blue"]})
sizes = pd.DataFrame({"size": ["S", "M", "L"]})

show_tables(colors, sizes)

,color
0,black
1,white
2,blue
,size
0,S
1,M
2,L


In [84]:
# 가능한 모든 조합
colors.merge(sizes, how="cross")

,color,size
0,black,S
1,black,M
2,black,L
3,white,S
4,white,M
5,white,L
6,blue,S
7,blue,M
8,blue,L


In [85]:
# 조합 개수 확인
len(colors.merge(sizes, how="cross"))

9

## 17. merge 후 결측값 처리

In [86]:
# left join 결과
customer_purchase = customers.merge(purchases, on="customer_id", how="left")
customer_purchase

,customer_id,name,region,product,price
0,1,김지민,서울,노트북,1.20e+06
1,2,이서연,경기,마우스,2.50e+04
2,2,이서연,경기,키보드,8.00e+04
3,3,박민수,서울,NaN,NaN
4,4,최하준,부산,NaN,NaN


In [87]:
# 결측값 확인
customer_purchase.isna().sum()

,0
customer_id,0
name,0
region,0
product,2
price,2


In [88]:
# 결측값 채우기
customer_purchase_filled = customer_purchase.fillna({
    "product": "구매없음",
    "price": 0
})
customer_purchase_filled

,customer_id,name,region,product,price
0,1,김지민,서울,노트북,1.20e+06
1,2,이서연,경기,마우스,2.50e+04
2,2,이서연,경기,키보드,8.00e+04
3,3,박민수,서울,구매없음,0.00e+00
4,4,최하준,부산,구매없음,0.00e+00


In [89]:
# 지역별 구매금액 합계
customer_purchase_filled.groupby("region").agg(
    고객수=("customer_id", "nunique"),
    구매금액=("price", "sum")
)

,고객수,구매금액
region,,
경기,1,1.05e+05
부산,1,0.00e+00
서울,2,1.20e+06


## 18. 실습 정리 코드

In [90]:
# 카테고리별 매출 상위 정리
category_summary = orders.groupby("category", as_index=False).agg(
    주문건수=("order_id", "count"),
    총수량=("qty", "sum"),
    총매출=("amount", "sum")
).sort_values("총매출", ascending=False)

category_summary

,category,주문건수,총수량,총매출
0,coffee,6,11,53500
2,tea,3,8,40000
1,dessert,3,5,32500


In [91]:
# 고객 정보와 구매 정보를 합친 뒤 지역별 요약
customers.merge(purchases, on="customer_id", how="left").fillna({
    "product": "구매없음",
    "price": 0
}).groupby("region").agg(
    고객수=("customer_id", "nunique"),
    구매건수=("product", lambda x: (x != "구매없음").sum()),
    총구매금액=("price", "sum")
)

,고객수,구매건수,총구매금액
region,,,
경기,1,2,1.05e+05
부산,1,0,0.00e+00
서울,2,1,1.20e+06


In [92]:
# 주문 헤더와 상세를 합친 뒤 고객별 수량 요약
order_header.merge(order_detail, on="order_id", how="left").groupby("customer").agg(
    주문수=("order_id", "nunique"),
    품목수=("item", "count"),
    총수량=("qty", "sum")
)

,주문수,품목수,총수량
customer,,,
민수,1,2,5
서연,1,1,1
지민,1,2,3
